In [ ]:
#Importar librerias
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# Leer el archivo
df = pd.read_csv('DataSetTechmindV1.csv')
df.head()

,id_fragmento,titulo_origen,categoria_l1,pagina,texto_crudo,longitud_caracteres,fecha_extraccion,hash_texto
0,Redes_y__Fundamentos_de_R_568f47_p001_001,Fundamentos de Redes y TCP.pdf,Redes_y_Comunicaciones,1,6 Interfaces de Programación de Aplicaciones (...,1425,2026-07-23T10:11:00.466220+00:00,44ff810c25cbe49d8fc55eb61799ff3118b92686f5cbc8...
1,Redes_y__Fundamentos_de_R_568f47_p002_002,Fundamentos de Redes y TCP.pdf,Redes_y_Comunicaciones,2,1.1 1.1 El Ecosistema de Códigos de Estado ( S...,2297,2026-07-23T10:11:00.475681+00:00,30bb68cc425db598b02dfdb63d8c2a58d4a285da4d0831...
2,Redes_y__Fundamentos_de_R_568f47_p003_003,Fundamentos de Redes y TCP.pdf,Redes_y_Comunicaciones,3,Código Denominación Explicación Arquitectónica...,1848,2026-07-23T10:11:00.478992+00:00,5aba7f5523eb920a3d2dd6e4c9515bc6b1fa92c482ec2b...
3,Redes_y__Fundamentos_de_R_568f47_p004_004,Fundamentos de Redes y TCP.pdf,Redes_y_Comunicaciones,4,Axioma de Ingeniería REST En el diseño de fram...,2818,2026-07-23T10:11:00.482425+00:00,2f6a2818de4523f0764e8330be108e6d812b55082f48ed...
4,Redes_y__Fundamentos_de_R_568f47_p005_005,Fundamentos de Redes y TCP.pdf,Redes_y_Comunicaciones,5,Navegador / Cliente HTTP Servidor Origen Nuevo...,1747,2026-07-23T10:11:00.492391+00:00,6ebf6523f2590c2efa8ec6208a4c74514c5e4ea9a318f8...


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 668 entries, 0 to 667
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id_fragmento         668 non-null    object
 1   titulo_origen        668 non-null    object
 2   categoria_l1         668 non-null    object
 3   pagina               668 non-null    int64 
 4   texto_crudo          668 non-null    object
 5   longitud_caracteres  668 non-null    int64 
 6   fecha_extraccion     668 non-null    object
 7   hash_texto           668 non-null    object
dtypes: int64(2), object(6)
memory usage: 41.9+ KB


In [ ]:
def limpiar(texto):
    if not isinstance(texto, str):
        return ""

    # 1. Se cambia el texto normalizado en minusculas
    texto = texto.lower()

    # 2. Se remplazan todos los saltos que se generan y las tabulaciones del archivo por espacios.
    texto = re.sub(r'[\n\t\r]+', ' ', texto)

    # 3. Eliminamos Urls en dado caso que existieran.
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto, flags=re.MULTILINE)

    # 4. Eliminamos los números del texto
    texto = re.sub(r'\d+', ' ', texto)

    # 5. Se eliminan los signos de puntuacion y los caracteres especiales.
    texto = re.sub(r'[^áéíóúüñA-Za-z0-9\s]', ' ', texto)

    # 6. Si existen mas de un espacio entre letras se elimina dejando uno.
    texto = re.sub(r'\s+', ' ', texto)

    # 7. Buscamos y eliminamos palabras que no correspandan
    palabras = [p for p in texto.split() if len(p) > 2]
    texto = " ".join(palabras)

    return texto.strip()

In [ ]:
df['texto_limpio']=df['texto_crudo'].apply(limpiar)

In [ ]:
df

,id_fragmento,titulo_origen,categoria_l1,pagina,texto_crudo,longitud_caracteres,fecha_extraccion,hash_texto,texto_limpio
0,Redes_y__Fundamentos_de_R_568f47_p001_001,Fundamentos de Redes y TCP.pdf,Redes_y_Comunicaciones,1,6 Interfaces de Programación de Aplicaciones (...,1425,2026-07-23T10:11:00.466220+00:00,44ff810c25cbe49d8fc55eb61799ff3118b92686f5cbc8...,interfaces programación aplicaciones apis defi...
1,Redes_y__Fundamentos_de_R_568f47_p002_002,Fundamentos de Redes y TCP.pdf,Redes_y_Comunicaciones,2,1.1 1.1 El Ecosistema de Códigos de Estado ( S...,2297,2026-07-23T10:11:00.475681+00:00,30bb68cc425db598b02dfdb63d8c2a58d4a285da4d0831...,ecosistema códigos estado status codes ciclo v...
2,Redes_y__Fundamentos_de_R_568f47_p003_003,Fundamentos de Redes y TCP.pdf,Redes_y_Comunicaciones,3,Código Denominación Explicación Arquitectónica...,1848,2026-07-23T10:11:00.478992+00:00,5aba7f5523eb920a3d2dd6e4c9515bc6b1fa92c482ec2b...,código denominación explicación arquitectónica...
3,Redes_y__Fundamentos_de_R_568f47_p004_004,Fundamentos de Redes y TCP.pdf,Redes_y_Comunicaciones,4,Axioma de Ingeniería REST En el diseño de fram...,2818,2026-07-23T10:11:00.482425+00:00,2f6a2818de4523f0764e8330be108e6d812b55082f48ed...,axioma ingeniería rest diseño frameworks moder...
4,Redes_y__Fundamentos_de_R_568f47_p005_005,Fundamentos de Redes y TCP.pdf,Redes_y_Comunicaciones,5,Navegador / Cliente HTTP Servidor Origen Nuevo...,1747,2026-07-23T10:11:00.492391+00:00,6ebf6523f2590c2efa8ec6208a4c74514c5e4ea9a318f8...,navegador cliente http servidor origen nuevo s...
...,...,...,...,...,...,...,...,...,...
663,ocr_Inteli_Prompt_AWS_904c5d_p14_15,Prompt AWS.pdf,Inteligencia_Artificial,14,FM de Amazon Titan: los modelos de Amazon Tita...,2245,2026-07-23T12:33:58.926707+00:00,7253b8911f7ef69819ac38b9dc153ad63aa04ac203edc0...,amazon titan los modelos amazon titan foundati...
664,ocr_Inteli_Prompt_AWS_904c5d_p16_18,Prompt AWS.pdf,Inteligencia_Artificial,16,Secuencias de parada\n\nStopSequences es una l...,1796,2026-07-23T12:33:58.926707+00:00,ab7f92e6d279caf249b87b80b9fdff2f15ae556a3181b6...,secuencias parada stopsequences una lista cade...
665,ocr_Inteli_Prompt_AWS_904c5d_p19_20,Prompt AWS.pdf,Inteligencia_Artificial,19,Los dates con los que se entrenan los modelos ...,1790,2026-07-23T12:33:58.926707+00:00,0de10f2d03cf9c9e40e13b381fbf28314c284188ee48a4...,los dates con los que entrenan los modelos pue...
666,ocr_Inteli_Prompt_AWS_904c5d_p21_22,Prompt AWS.pdf,Inteligencia_Artificial,21,LLANKIFILAR US AML EL APRENDIZAJE DE Few-Shot\...,1798,2026-07-23T12:33:58.926707+00:00,ea06b947206ec3cbacffa86ec727a33a45b21395dda9b4...,llankifilar aml aprendizaje few shot marco des...


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 668 entries, 0 to 667
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id_fragmento         668 non-null    object
 1   titulo_origen        668 non-null    object
 2   categoria_l1         668 non-null    object
 3   pagina               668 non-null    int64 
 4   texto_crudo          668 non-null    object
 5   longitud_caracteres  668 non-null    int64 
 6   fecha_extraccion     668 non-null    object
 7   hash_texto           668 non-null    object
 8   texto_limpio         668 non-null    object
dtypes: int64(2), object(7)
memory usage: 47.1+ KB


In [ ]:
df['texto_limpio']=df['texto_limpio'].astype('string')

IT-IDF

In [ ]:
# Se define los stop words con todas las uniones posibles para aplicar el modelo
stopwords = [
    # Artículos y determinantes
    'el', 'la', 'los', 'las', 'un', 'una', 'unos', 'unas', 'lo', 'al', 'del',
    # Preposiciones
    'a', 'ante', 'bajo', 'cabe', 'con', 'contra', 'de', 'desde', 'durante',
    'en', 'entre', 'hacia', 'hasta', 'mediante', 'para', 'por', 'según',
    'sin','sobre', 'tras', 'vía',
    # Conjunciones
    'y', 'e', 'ni', 'o', 'u', 'pero', 'sino', 'porque', 'que', 'si', 'aunque',
    'como', 'cuando', 'donde', 'mientras', 'que',
    # Pronombres
    'yo', 'tú', 'él', 'ella', 'nosotros', 'nosotras',
    'ellos', 'ellas', 'me', 'te', 'se', 'nos', 'mi', 'tu', 'su', 'mis',
    'tus', 'sus', 'mío', 'tuyo', 'suyo', 'este', 'esta', 'estos', 'estas',
    'ese', 'esa', 'esos', 'esas', 'aquel', 'aquella', 'aquellos', 'aquellas',
    'quien', 'quienes', 'cuyo', 'cuyos', 'cuya', 'cuyas', 'cual', 'cuales',
    # Verbos auxiliares
    'es', 'son', 'fue', 'fueron', 'sea', 'sean', 'será', 'serán', 'sería',
    'estar', 'está', 'están', 'estaba', 'estaban', 'estado', 'hay', 'había',
    'han', 'has', 'he', 'hayan', 'haya', 'tener', 'tiene', 'tienen', 'tuvo',
    'hacer', 'hace', 'hizo', 'hacen', 'decir', 'dice', 'van', 'va', 'fue',
    # Adverbios y otros conectores frecuentes
    'no', 'sí', 'ya', 'muy', 'más', 'menos', 'también', 'tampoco', 'tan',
    'tanto', 'todo', 'todos', 'toda', 'todas', 'otro', 'otra', 'otros',
    'otras', 'alguno', 'alguna', 'algunos', 'algunas', 'ninguno', 'ninguna',
    'poco', 'poca', 'pocos', 'pocas', 'mucho', 'mucha', 'muchos', 'muchas',
    'cada', 'cualquier', 'quienquiera', 'tal', 'tales', 'asi', 'así', 'aquí',
    'allí', 'ahí', 'donde', 'cuando', 'como', 'porque', 'entonces', 'además'
]

In [ ]:
#Aplicamos los stopwords al vector
vector = TfidfVectorizer(stop_words=stopwords)

In [ ]:
#se aplica a la columna que se limpio en el modelo
X_tfidf = vector.fit_transform(df['texto_limpio'])
print(f'Dimenciones: {X_tfidf.shape}')

Dimenciones: (668, 16283)


REGRESION LOGISTICA

In [ ]:
# Se define (X) con el modelo aplicado con TF-IDF
# Definimos (y) con la columna de categoria con el fin de hacer la prediccion
X = X_tfidf
y = df['categoria_l1']


In [ ]:
# Aqui vamos a dividr los datos de tal forma que con el 80% vamos a entrenar el modelo y el con el 20% se prueba
X_train, X_test, y_train, y_test, = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# se crea y se inicia el modelo de regresion logistica
modelo = LogisticRegression()

In [ ]:
# Entrenamiento con los datos que usamos para entrenar el modelo
modelo.fit(X_train, y_train)

LogisticRegression()

In [ ]:
# Predicciones usando los datos de prueba
y_pred = modelo.predict(X_test)

In [ ]:
# Hacemos evaluaciones
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
print(classification_report(y_test, y_pred))

Accuracy: 0.8656716417910447
                         precision    recall  f1-score   support

           Arquitectura       0.54      0.93      0.68        14
         Bases_de_Datos       1.00      0.56      0.71         9
               Hardware       1.00      1.00      1.00        14
Inteligencia_Artificial       0.93      0.93      0.93        30
 Lenguajes_Programacion       0.92      0.94      0.93        36
 Redes_y_Comunicaciones       0.91      0.95      0.93        22
    Sistemas_Operativos       1.00      0.11      0.20         9

               accuracy                           0.87       134
              macro avg       0.90      0.78      0.77       134
           weighted avg       0.90      0.87      0.85       134



Con estos primeros datos se observa que para los temas de Hardware, inteligencia artificial, lenguajes de programacion, redes y comunicaciones tiene un F1-score alto por lo tanto el modelo aprendio a identificar con mayor precicion estos temas.

En Arquitectura tiene alto recall y baja precicion, probablemente el modelo esta confundiendo otras cosas con arquitectura metiendo textos de otras categorias por error

